In [ ]:
# Mount Google Drive and initialize Logistic Regression experiment paths

from google.colab import drive
from pathlib import Path

import json
import random
import time
import hashlib
import joblib
import platform
import sys
import gc
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.exceptions import ConvergenceWarning

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '02_logistic_regression'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

MODELS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'logistic_regression'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    MODELS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Load shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']
FEATURE_BUDGET = config['classical_features']['feature_budget']

MODEL_NAME = 'Logistic Regression'

REPRESENTATIONS = [
    'word_tfidf',
    'character_tfidf',
    'combined_tfidf'
]

DEFAULT_THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)

assert config['classical_features']['status'] == 'verified'
assert REGISTRY_PATH.is_file()

assert all(
    representation in config['classical_features']['representations']
    for representation in REPRESENTATIONS
)

# Reuse the verified evaluation module

sys.path.insert(0, str(SRC_DIR))

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

def calculate_sha256(path):
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    return sha256.hexdigest()

# Verify consistency with the previous Naive Bayes experiment

registry_df = pd.read_csv(REGISTRY_PATH)

assert registry_df['run_id'].is_unique

nb_record = registry_df.loc[
    registry_df['run_id'] == 'NB_WORD_BASELINE_S42'
].iloc[0]

assert nb_record['status'] == 'Completed'

with open(
    PROJECT_ROOT / nb_record['config_path'], 'r'
) as file:
    nb_run_config = json.load(file)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

assert evaluation_hash == nb_run_config['evaluation_module_sha256']

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == nb_run_config['feature_manifest_sha256']

# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

REPRESENTATION_COLORS = {
    'word_tfidf': '#4C78A8',
    'character_tfidf': '#F58518',
    'combined_tfidf': '#54A24B'
}

# Record the current environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

display(environment_df)

# Verify the feature extraction environment compatibility

feature_environment_path = (
    CLASSICAL_DIR / '00_feature_extraction'
    / 'results' / 'tables' / 'environment_versions.csv'
)

feature_environment_df = pd.read_csv(feature_environment_path)

environment_comparison_df = environment_df.merge(
    feature_environment_df,
    on='Package',
    suffixes=('_Current', '_Feature_Extraction')
)

environment_comparison_df['Version Match'] = (
    environment_comparison_df['Version_Current']
    == environment_comparison_df['Version_Feature_Extraction']
)

display(environment_comparison_df)

environment_comparison_df.to_csv(
    TABLES_DIR / 'environment_compatibility.csv',
    index=False
)

print("\nNotebook:", NOTEBOOK_DIR.name)
print("Model:", MODEL_NAME)
print("Feature representations:", REPRESENTATIONS)
print("Common feature budget:", FEATURE_BUDGET)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("Results directory:", RESULTS_DIR)
print("Model directory:", MODELS_DIR)
print("Previously completed experiments:",
      registry_df['status'].eq('Completed').sum())

print("Shared evaluation module: Verified")
print("Feature artifact manifest: Verified")
print("Official test dataset: Not loaded")

print("\nLogistic Regression environment initialized successfully.")

In [ ]:
# Load and verify the fixed training and validation datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

datasets = {}
dataset_records = {}

verification_records = []

for partition, source_partition in [
    ('train', 'train_representative'),
    ('validation', 'validation')
]:

    record = dataset_manifest.loc[
        dataset_manifest['Partition'] == source_partition
    ].iloc[0]

    dataset_path = PROJECT_ROOT / record['Dataset Path']

    dataset_hash = calculate_sha256(dataset_path)

    assert dataset_hash == record['SHA256']

    df = pd.read_parquet(dataset_path)

    assert len(df) == record['Rows']
    assert df['id'].is_unique
    assert df[LABELS].isin([0, 1]).all().all()
    assert df['comment_text'].notna().all()

    # Verify feature row alignment

    row_record = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Row Manifest')
        & (feature_manifest['Partition'] == partition)
    ].iloc[0]

    row_path = PROJECT_ROOT / row_record['Path']

    assert calculate_sha256(row_path) == row_record['SHA256']
    assert row_record['Source Dataset SHA256'] == dataset_hash

    row_manifest = pd.read_csv(
        row_path,
        dtype={'id': str}
    )

    assert row_manifest['id'].tolist() == df['id'].tolist()

    assert (
        row_manifest['group_id'].tolist()
        == df['group_id'].tolist()
    )

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    datasets[partition] = df
    dataset_records[partition] = record

    verification_records.append({
        'Partition': partition,
        'Comments': len(df),
        'Unique IDs': df['id'].nunique(),
        'Dataset Hash Verified': True,
        'Row Manifest Verified': True,
        'Row Alignment Verified': True,
        'Valid Labels': True
    })

train_df = datasets['train']
val_df = datasets['validation']

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_val = val_df[LABELS].to_numpy(dtype=np.int8)

assert set(train_df['id']).isdisjoint(val_df['id'])
assert set(train_df['group_id']).isdisjoint(val_df['group_id'])

# Verify consistency with the Naive Bayes experiment

assert (
    dataset_records['train']['SHA256']
    == nb_run_config['training_dataset_sha256']
)

assert (
    dataset_records['validation']['SHA256']
    == nb_run_config['validation_dataset_sha256']
)

dataset_verification_df = pd.DataFrame(
    verification_records
)

display(dataset_verification_df)

dataset_verification_df.to_csv(
    TABLES_DIR / 'dataset_verification.csv',
    index=False
)

# Define a reusable feature-loading function

def load_verified_features(representation):

    assert representation in REPRESENTATIONS

    matrices = {}
    records = []

    for partition in ['train', 'validation']:

        record = feature_manifest.loc[
            (feature_manifest['Artifact Type'] == 'Sparse Matrix')
            & (feature_manifest['Representation'] == representation)
            & (feature_manifest['Partition'] == partition)
        ].iloc[0]

        path = PROJECT_ROOT / record['Path']

        assert calculate_sha256(path) == record['SHA256']

        assert (
            record['Source Dataset SHA256']
            == dataset_records[partition]['SHA256']
        )

        matrix = sp.load_npz(path)

        assert matrix.format == 'csr'
        assert matrix.dtype == np.float32

        assert matrix.shape == (
            len(datasets[partition]),
            FEATURE_BUDGET
        )

        assert np.isfinite(matrix.data).all()
        assert (matrix.data >= 0).all()

        memory_bytes = (
            matrix.data.nbytes
            + matrix.indices.nbytes
            + matrix.indptr.nbytes
        )

        matrices[partition] = matrix

        records.append({
            'Representation': representation,
            'Partition': partition,
            'Comments': matrix.shape[0],
            'Features': matrix.shape[1],
            'Nonzero Elements': matrix.nnz,
            'Data Type': str(matrix.dtype),
            'Sparse Memory (MiB)': memory_bytes / (1024 ** 2),
            'Zero Feature Comments':
                int(np.sum(np.diff(matrix.indptr) == 0)),
            'Feature Hash Verified': True,
            'Source Dataset Verified': True,
            'Verification': 'Passed'
        })

    return (
        matrices['train'],
        matrices['validation'],
        pd.DataFrame(records)
    )


# Verify all representations sequentially

feature_verification_records = []

for representation in REPRESENTATIONS:

    print(f"Verifying {representation}...")

    X_train_temp, X_val_temp, verification = (
        load_verified_features(representation)
    )

    feature_verification_records.append(verification)

    print(
        "Training:", X_train_temp.shape,
        "| Validation:", X_val_temp.shape
    )

    del X_train_temp, X_val_temp
    gc.collect()

feature_verification_df = pd.concat(
    feature_verification_records,
    ignore_index=True
)

display(feature_verification_df.round(4))

feature_verification_df.to_csv(
    TABLES_DIR / 'feature_representation_verification.csv',
    index=False
)

# Record label support for the shared experimental partitions

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Validation Negatives': len(y_val) - y_val.sum(axis=0)
})

label_support_df['Training Imbalance Ratio'] = (
    label_support_df['Training Negatives']
    / label_support_df['Training Positives']
)

display(label_support_df.round(4))

label_support_df.to_csv(
    TABLES_DIR / 'label_support_and_imbalance.csv',
    index=False
)

print("\nTraining label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)
print("Feature representations verified:", len(REPRESENTATIONS))
print("Sparse matrices verified:", len(feature_verification_df))
print("Consistency with Naive Bayes dataset: Verified")
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Official test dataset: Not loaded")
print("Reusable feature loader: Ready")
print("\nDataset and feature verification completed successfully.")

In [ ]:
# Define common Logistic Regression hyperparameters

LR_PARAMS = {
    'solver': 'liblinear',
    'penalty': 'l2',
    'C': 1.0,
    'dual': False,
    'tol': 1e-4,
    'max_iter': 1000,
    'fit_intercept': True,
    'intercept_scaling': 1.0,
    'class_weight': None,
    'random_state': SEED
}

OVR_PARAMS = {
    'n_jobs': 1
}

# Define the controlled representation experiments

experiment_specs = [
    {
        'run_id': 'LR_WORD_BASELINE_S42',
        'representation': 'word_tfidf',
        'class_weight': None,
        'purpose': 'Common word-based baseline'
    },
    {
        'run_id': 'LR_CHAR_BASELINE_S42',
        'representation': 'character_tfidf',
        'class_weight': None,
        'purpose': 'Character representation comparison'
    },
    {
        'run_id': 'LR_COMBINED_BASELINE_S42',
        'representation': 'combined_tfidf',
        'class_weight': None,
        'purpose': 'Combined representation comparison'
    }
]

experiment_plan_df = pd.DataFrame(experiment_specs)

experiment_plan_df['Training Samples'] = len(y_train)
experiment_plan_df['Validation Samples'] = len(y_val)
experiment_plan_df['Features'] = FEATURE_BUDGET
experiment_plan_df['Seed'] = SEED
experiment_plan_df['Status'] = 'Planned'

print("Logistic Regression experiment plan")

display(experiment_plan_df)

# Document the hyperparameter selection rationale

parameter_rationale = {
    'solver': (
        'Suitable for sparse binary classification with '
        'L2 regularization.'
    ),
    'penalty': (
        'Controls coefficient magnitude and reduces '
        'overfitting in high-dimensional feature spaces.'
    ),
    'C': (
        'Standard initial inverse regularization strength. '
        'Held constant during representation comparisons.'
    ),
    'dual': (
        'Primal formulation selected because the number of '
        'training samples exceeds the number of features.'
    ),
    'tol': 'Standard optimization convergence tolerance.',
    'max_iter': (
        'Generous iteration limit. Actual convergence '
        'will be verified after fitting.'
    ),
    'fit_intercept': 'Allows a nonzero decision boundary offset.',
    'intercept_scaling': (
        'Standard liblinear intercept scaling. '
        'The synthetic intercept feature is regularized.'
    ),
    'class_weight': (
        'No weighting in the initial representation '
        'comparison. Balanced weighting evaluated separately.'
    ),
    'random_state': 'Maintains the common experimental seed.'
}

hyperparameter_df = pd.DataFrame([
    {
        'Parameter': parameter,
        'Value': str(value),
        'Justification': parameter_rationale[parameter]
    }
    for parameter, value in LR_PARAMS.items()
])

display(hyperparameter_df)

# Calculate expected balanced class weights from training labels

positive_counts = y_train.sum(axis=0)
negative_counts = len(y_train) - positive_counts

assert (positive_counts > 0).all()
assert (negative_counts > 0).all()

negative_weights = len(y_train) / (2 * negative_counts)
positive_weights = len(y_train) / (2 * positive_counts)

class_weight_df = pd.DataFrame({
    'Label': LABELS,
    'Negative Support': negative_counts,
    'Positive Support': positive_counts,
    'Negative Class Weight': negative_weights,
    'Positive Class Weight': positive_weights,
    'Positive to Negative Weight Ratio':
        positive_weights / negative_weights
})

print("\nExpected balanced class weights")
display(class_weight_df.round(6))

# Verify equal total weighted support for both classes

assert np.allclose(
    negative_weights * negative_counts,
    len(y_train) / 2
)

assert np.allclose(
    positive_weights * positive_counts,
    len(y_train) / 2
)

# Create a common model constructor

def build_lr_model(class_weight=None):

    parameters = LR_PARAMS.copy()
    parameters['class_weight'] = class_weight

    return OneVsRestClassifier(
        LogisticRegression(**parameters),
        **OVR_PARAMS
    )

lr_template = build_lr_model()

assert lr_template.estimator.get_params()['C'] == 1.0
assert lr_template.estimator.get_params()['solver'] == 'liblinear'

# Save the common experiment configuration

lr_experiment_config = {
    'notebook': '03_classical_ml/02_logistic_regression',
    'model': MODEL_NAME,
    'classification_strategy': 'One-vs-Rest',
    'labels': LABELS,
    'num_labels': len(LABELS),
    'seed': SEED,
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'sampling_strategy': 'representative',
    'feature_budget': FEATURE_BUDGET,
    'model_parameters': LR_PARAMS,
    'wrapper_parameters': OVR_PARAMS,
    'experiments': experiment_specs,
    'score_type': 'positive_class_probability',
    'default_threshold': DEFAULT_THRESHOLD,
    'primary_selection_metric': 'macro_average_precision',
    'threshold_optimization_metric': 'per_label_f1',
    'threshold_tie_breaking': 'highest_threshold',
    'weighted_experiment': {
        'class_weight': 'balanced',
        'representation_selection': 'validation_macro_ap',
        'other_parameters': 'unchanged',
        'status': 'planned'
    },
    'training_dataset_sha256':
        dataset_records['train']['SHA256'],
    'validation_dataset_sha256':
        dataset_records['validation']['SHA256'],
    'feature_manifest_sha256':
        calculate_sha256(
            FEATURES_DIR / 'feature_artifact_manifest.csv'
        ),
    'evaluation_module_sha256': evaluation_hash,
    'sklearn_version': sklearn.__version__,
    'official_test_used': False
}

LR_EXPERIMENT_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'lr_experiment_config.json'
)

with open(LR_EXPERIMENT_CONFIG_PATH, 'w') as file:
    json.dump(lr_experiment_config, file, indent=4)

experiment_plan_df.to_csv(
    TABLES_DIR / 'lr_experiment_plan.csv',
    index=False
)

hyperparameter_df.to_csv(
    TABLES_DIR / 'lr_hyperparameter_rationale.csv',
    index=False
)

class_weight_df.to_csv(
    TABLES_DIR / 'lr_expected_balanced_class_weights.csv',
    index=False
)

print("\nLogistic Regression configuration summary")
print("Unweighted representation experiments:",
      len(experiment_specs))
print("Planned weighted experiments: 1")
print("Common solver:", LR_PARAMS['solver'])
print("Regularization:", LR_PARAMS['penalty'])
print("Regularization strength C:", LR_PARAMS['C'])
print("Maximum iterations:", LR_PARAMS['max_iter'])
print("Model selection metric: Macro AP")
print("Threshold optimization: Per-label validation F1")
print("Balanced class weights: Verified")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nLogistic Regression configuration completed successfully.")

In [ ]:
# Define a reusable training and evaluation pipeline

def run_lr_experiment(spec):

    run_id = spec['run_id']
    representation = spec['representation']
    class_weight = spec['class_weight']

    print(f"\nStarting experiment: {run_id}")
    print("Representation:", representation)
    print("Class weight:", class_weight)

    # Create separate output directories for each run

    run_dir = RESULTS_DIR / run_id
    run_tables_dir = run_dir / 'tables'
    run_predictions_dir = run_dir / 'predictions'
    run_logs_dir = run_dir / 'logs'

    for folder in [
        run_tables_dir,
        run_predictions_dir,
        run_logs_dir
    ]:
        folder.mkdir(parents=True, exist_ok=True)

    model_path = MODELS_DIR / f'{run_id}.joblib'
    config_path = RUN_CONFIGS_DIR / f'{run_id}.json'
    threshold_path = RUN_CONFIGS_DIR / f'{run_id}_thresholds.json'

    # Load the verified feature representation

    load_start = time.perf_counter()

    X_train, X_val, feature_checks = load_verified_features(
        representation
    )

    feature_loading_time = time.perf_counter() - load_start

    print("Training features:", X_train.shape)
    print("Validation features:", X_val.shape)

    model = build_lr_model(class_weight=class_weight)

    # Train and capture convergence warnings

    training_start = datetime.now(timezone.utc).isoformat()
    start = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:

        warnings.simplefilter('always', ConvergenceWarning)

        model.fit(X_train, y_train)

    training_time = time.perf_counter() - start
    training_end = datetime.now(timezone.utc).isoformat()

    convergence_warnings = [
        str(warning.message)
        for warning in captured_warnings
        if issubclass(warning.category, ConvergenceWarning)
    ]

    warning_records = [
        {
            'category': warning.category.__name__,
            'message': str(warning.message)
        }
        for warning in captured_warnings
    ]

    with open(
        run_logs_dir / 'training_warnings.json', 'w'
    ) as file:
        json.dump(warning_records, file, indent=4)

    assert not convergence_warnings, (
        f"Convergence warnings detected in {run_id}"
    )

    # Verify all six fitted binary classifiers

    assert model.multilabel_
    assert len(model.estimators_) == len(LABELS)

    assert np.array_equal(
        model.classes_,
        np.arange(len(LABELS))
    )

    training_records = []

    for i, estimator in enumerate(model.estimators_):

        assert np.array_equal(estimator.classes_, [0, 1])
        assert estimator.coef_.shape == (1, FEATURE_BUDGET)
        assert np.isfinite(estimator.coef_).all()
        assert np.isfinite(estimator.intercept_).all()

        iterations = int(estimator.n_iter_.max())

        assert iterations < LR_PARAMS['max_iter']

        training_records.append({
            'Label': LABELS[i],
            'Iterations': iterations,
            'Maximum Iterations': LR_PARAMS['max_iter'],
            'Coefficient L2 Norm':
                float(np.linalg.norm(estimator.coef_)),
            'Intercept': float(estimator.intercept_[0]),
            'Convergence': 'Passed'
        })

    training_df = pd.DataFrame(training_records)

    print("\nClassifier training summary")
    display(training_df.round(6))

    # Generate validation probability scores

    start = time.perf_counter()

    val_scores = model.predict_proba(X_val)

    prediction_time = time.perf_counter() - start

    assert val_scores.shape == y_val.shape
    assert np.isfinite(val_scores).all()
    assert ((val_scores >= 0) & (val_scores <= 1)).all()

    # Verify positive-class score alignment

    for i, estimator in enumerate(model.estimators_):

        expected_scores = estimator.predict_proba(
            X_val[:100]
        )[:, 1]

        assert np.allclose(
            val_scores[:100, i],
            expected_scores
        )

    # Evaluate the default threshold

    default_metrics, default_per_label_df = evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=DEFAULT_THRESHOLD
    )

    default_predictions = apply_thresholds(
        val_scores,
        DEFAULT_THRESHOLD
    )

    # Optimize thresholds using validation data only

    threshold_df = optimize_thresholds(
        y_val,
        val_scores,
        LABELS
    )

    assert threshold_df['Label'].tolist() == LABELS

    optimized_thresholds = threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    assert np.isfinite(optimized_thresholds).all()
    assert ((optimized_thresholds >= 0) &
            (optimized_thresholds <= 1)).all()

    optimized_metrics, optimized_per_label_df = evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=optimized_thresholds
    )

    optimized_predictions = apply_thresholds(
        val_scores,
        optimized_thresholds
    )

    # Verify the threshold optimization results

    assert np.allclose(
        threshold_df['Validation F1'],
        optimized_per_label_df['F1']
    )

    assert np.all(
        optimized_per_label_df['F1'].to_numpy()
        >= default_per_label_df['F1'].to_numpy() - 1e-10
    )

    for metric in [
        'Macro AP',
        'Micro AP',
        'Macro ROC-AUC',
        'Micro ROC-AUC'
    ]:
        assert np.isclose(
            default_metrics[metric],
            optimized_metrics[metric]
        )

    # Construct the comparison tables

    overall_comparison_df = pd.DataFrame([
        {
            'Metric': metric,
            'Default': default_metrics[metric],
            'Optimized': optimized_metrics[metric],
            'Change':
                optimized_metrics[metric] - default_metrics[metric]
        }
        for metric in default_metrics
    ])

    per_label_comparison_df = pd.DataFrame({
        'Label': LABELS,
        'Average Precision':
            default_per_label_df['Average Precision'],
        'ROC-AUC': default_per_label_df['ROC-AUC'],
        'Default Threshold': DEFAULT_THRESHOLD,
        'Optimized Threshold': optimized_thresholds,
        'Default Precision': default_per_label_df['Precision'],
        'Optimized Precision': optimized_per_label_df['Precision'],
        'Default Recall': default_per_label_df['Recall'],
        'Optimized Recall': optimized_per_label_df['Recall'],
        'Default F1': default_per_label_df['F1'],
        'Optimized F1': optimized_per_label_df['F1'],
        'Actual Positives': optimized_per_label_df['Positive Support']
    })

    per_label_comparison_df['F1 Improvement'] = (
        per_label_comparison_df['Optimized F1']
        - per_label_comparison_df['Default F1']
    )

    # Save the fitted model and verify its reconstruction

    joblib.dump(model, model_path, compress=3)

    reloaded_model = joblib.load(model_path)

    reloaded_scores = reloaded_model.predict_proba(X_val)

    maximum_score_error = float(
        np.abs(reloaded_scores - val_scores).max()
    )

    assert np.allclose(
        reloaded_scores,
        val_scores,
        rtol=0,
        atol=1e-12
    )

    assert np.array_equal(
        apply_thresholds(reloaded_scores, DEFAULT_THRESHOLD),
        default_predictions
    )

    assert np.array_equal(
        apply_thresholds(reloaded_scores, optimized_thresholds),
        optimized_predictions
    )

    # Save ID-aligned validation scores and predictions

    predictions_df = pd.DataFrame({
        'id': val_df['id'].to_numpy()
    })

    for i, label in enumerate(LABELS):

        predictions_df[f'{label}_true'] = y_val[:, i]
        predictions_df[f'{label}_score'] = val_scores[:, i]

        predictions_df[f'{label}_pred_default'] = (
            default_predictions[:, i]
        )

        predictions_df[f'{label}_pred_optimized'] = (
            optimized_predictions[:, i]
        )

    predictions_path = (
        run_predictions_dir / 'validation_predictions.parquet'
    )

    predictions_df.to_parquet(
        predictions_path,
        index=False
    )

    saved_predictions_df = pd.read_parquet(predictions_path)

    pd.testing.assert_frame_equal(
        predictions_df,
        saved_predictions_df
    )

    assert saved_predictions_df['id'].is_unique

    # Save full-precision thresholds

    threshold_config = {
        'run_id': run_id,
        'selection_partition': 'validation',
        'optimization_metric': 'per_label_f1',
        'comparison_operator': 'greater_than_or_equal',
        'tie_breaking': 'highest_threshold',
        'score_type': 'positive_class_probability',
        'default_threshold': DEFAULT_THRESHOLD,
        'labels': LABELS,
        'thresholds': {
            label: float(threshold)
            for label, threshold in zip(
                LABELS, optimized_thresholds
            )
        },
        'official_test_used': False
    }

    with open(threshold_path, 'w') as file:
        json.dump(threshold_config, file, indent=4)

    with open(threshold_path, 'r') as file:
        saved_threshold_config = json.load(file)

    saved_thresholds = np.array([
        saved_threshold_config['thresholds'][label]
        for label in LABELS
    ])

    assert np.array_equal(
        saved_thresholds,
        optimized_thresholds
    )

    # Recompute metrics using saved validation artifacts

    saved_targets = saved_predictions_df[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    saved_scores = saved_predictions_df[[
        f'{label}_score' for label in LABELS
    ]].to_numpy()

    reconstructed_metrics, _ = evaluate_multilabel(
        saved_targets,
        saved_scores,
        LABELS,
        thresholds=saved_thresholds
    )

    for metric in optimized_metrics:

        assert np.isclose(
            reconstructed_metrics[metric],
            optimized_metrics[metric],
            rtol=0,
            atol=1e-12
        )

    # Record the individual run configuration

    model_parameters = LR_PARAMS.copy()
    model_parameters['class_weight'] = class_weight

    run_config = {
        'run_id': run_id,
        'notebook': '03_classical_ml/02_logistic_regression',
        'model': MODEL_NAME,
        'feature_representation': representation,
        'feature_count': FEATURE_BUDGET,
        'classification_strategy': 'One-vs-Rest',
        'labels': LABELS,
        'seed': SEED,
        'sampling_strategy': 'representative',
        'training_samples': len(y_train),
        'validation_samples': len(y_val),
        'model_parameters': model_parameters,
        'wrapper_parameters': OVR_PARAMS,
        'score_type': 'positive_class_probability',
        'primary_selection_metric': 'macro_average_precision',
        'threshold_optimization_metric': 'per_label_f1',
        'training_dataset_sha256':
            dataset_records['train']['SHA256'],
        'validation_dataset_sha256':
            dataset_records['validation']['SHA256'],
        'feature_manifest_sha256':
            lr_experiment_config['feature_manifest_sha256'],
        'evaluation_module_sha256': evaluation_hash,
        'threshold_path':
            str(threshold_path.relative_to(PROJECT_ROOT)),
        'model_path':
            str(model_path.relative_to(PROJECT_ROOT)),
        'sklearn_version': sklearn.__version__,
        'official_test_used': False
    }

    with open(config_path, 'w') as file:
        json.dump(run_config, file, indent=4)

    # Save numerical evaluation results

    tables = {
        'feature_verification.csv': feature_checks,
        'classifier_training_summary.csv': training_df,
        'default_overall_metrics.csv':
            pd.DataFrame([default_metrics]),
        'optimized_overall_metrics.csv':
            pd.DataFrame([optimized_metrics]),
        'default_per_label_metrics.csv': default_per_label_df,
        'optimized_per_label_metrics.csv': optimized_per_label_df,
        'optimized_thresholds.csv': threshold_df,
        'overall_threshold_comparison.csv': overall_comparison_df,
        'per_label_threshold_comparison.csv': per_label_comparison_df
    }

    for filename, dataframe in tables.items():

        dataframe.to_csv(
            run_tables_dir / filename,
            index=False
        )

    # Construct a consolidated experiment summary

    run_summary = {
        'Run ID': run_id,
        'Model': MODEL_NAME,
        'Feature Representation': representation,
        'Class Weight':
            class_weight if class_weight is not None else 'None',
        'Training Samples': len(y_train),
        'Validation Samples': len(y_val),
        'Features': FEATURE_BUDGET,
        'Macro AP': optimized_metrics['Macro AP'],
        'Micro AP': optimized_metrics['Micro AP'],
        'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
        'Default Macro F1': default_metrics['Macro F1'],
        'Optimized Macro F1': optimized_metrics['Macro F1'],
        'Default Micro F1': default_metrics['Micro F1'],
        'Optimized Micro F1': optimized_metrics['Micro F1'],
        'Optimized Macro Precision':
            optimized_metrics['Macro Precision'],
        'Optimized Macro Recall':
            optimized_metrics['Macro Recall'],
        'Training Time (s)': training_time,
        'Validation Prediction Time (s)': prediction_time,
        'Feature Loading Time (s)': feature_loading_time,
        'Model Size (MiB)':
            model_path.stat().st_size / (1024 ** 2),
        'Maximum Score Reconstruction Error': maximum_score_error,
        'Convergence Warnings': len(convergence_warnings),
        'Official Test Used': False,
        'Status': 'Completed'
    }

    pd.DataFrame([run_summary]).to_csv(
        run_tables_dir / 'run_summary.csv',
        index=False
    )

    # Save training and verification information

    training_log = {
        'run_id': run_id,
        'start_time': training_start,
        'end_time': training_end,
        'training_time_seconds': training_time,
        'prediction_time_seconds': prediction_time,
        'convergence_warnings': convergence_warnings,
        'maximum_score_reconstruction_error': maximum_score_error,
        'model_reload_verified': True,
        'prediction_alignment_verified': True,
        'metric_reconstruction_verified': True,
        'official_test_used': False
    }

    log_path = run_logs_dir / 'training_log.json'

    with open(log_path, 'w') as file:
        json.dump(training_log, file, indent=4)

    # Fingerprint critical experiment artifacts

    artifact_paths = {
        'Model': model_path,
        'Configuration': config_path,
        'Thresholds': threshold_path,
        'Validation Predictions': predictions_path,
        'Training Log': log_path,
        'Feature Manifest':
            FEATURES_DIR / 'feature_artifact_manifest.csv',
        'Evaluation Module':
            SRC_DIR / 'evaluation_utils.py'
    }

    artifact_manifest_df = pd.DataFrame([
        {
            'Run ID': run_id,
            'Artifact Type': artifact_type,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Size Bytes': path.stat().st_size,
            'SHA256': calculate_sha256(path)
        }
        for artifact_type, path in artifact_paths.items()
    ])

    artifact_manifest_df.to_csv(
        run_tables_dir / 'artifact_manifest.csv',
        index=False
    )

    # Prepare the registry record for later registration

    registry_record = {
        'run_id': run_id,
        'notebook': '03_classical_ml/02_logistic_regression',
        'model': MODEL_NAME,
        'feature_representation': representation,
        'loss_function': 'Binary logistic loss',
        'sampling_strategy': 'representative',
        'training_samples': len(y_train),
        'seed': SEED,
        'max_length': None,
        'config_path': str(config_path.relative_to(PROJECT_ROOT)),
        'model_path': str(model_path.relative_to(PROJECT_ROOT)),
        'results_path': str(run_dir.relative_to(PROJECT_ROOT)),
        'validation_macro_ap': optimized_metrics['Macro AP'],
        'validation_macro_f1': optimized_metrics['Macro F1'],
        'training_time_seconds': training_time,
        'status': 'Completed',
        'start_time': training_start,
        'end_time': training_end,
        'notes': (
            f"Class weight={class_weight}; "
            "L2 regularization; C=1.0; liblinear. "
            "Macro F1 uses validation-optimized thresholds. "
            "Model reload verified. Official test not used."
        )
    }

    # Display experiment results

    print("\nOverall validation performance")
    display(overall_comparison_df.round(6))

    print("\nPer-label performance comparison")
    display(per_label_comparison_df.round(6))

    print("\nExperiment summary")
    display(pd.DataFrame([run_summary]).T)

    print("\nModel reconstruction: Passed")
    print("Metric reconstruction: Passed")
    print("Experiment artifacts saved:", run_dir)
    print("Experiment completed:", run_id)

    # Retain only compact results in memory

    result = {
        'spec': spec.copy(),
        'summary': run_summary,
        'default_metrics': default_metrics,
        'optimized_metrics': optimized_metrics,
        'default_per_label': default_per_label_df,
        'optimized_per_label': optimized_per_label_df,
        'thresholds': threshold_df,
        'registry_record': registry_record,
        'predictions_path': predictions_path,
        'model_path': model_path,
        'results_path': run_dir
    }

    del X_train, X_val
    del model, reloaded_model
    del reloaded_scores, val_scores
    del predictions_df, saved_predictions_df

    gc.collect()

    return result


# Initialize the experiment result collection

lr_results = {}

assert callable(run_lr_experiment)

print("Reusable Logistic Regression pipeline: Ready")
print("Planned unweighted experiments:", len(experiment_specs))
print("Planned weighted experiments: 1")
print("Model training performed: No")
print("Official test dataset: Not loaded")

In [ ]:
# Train and evaluate the unweighted word TF-IDF baseline

word_spec = experiment_specs[0]

assert word_spec['representation'] == 'word_tfidf'
assert word_spec['class_weight'] is None

word_result = run_lr_experiment(word_spec)

lr_results[word_spec['run_id']] = word_result

# Compare Logistic Regression with the completed Naive Bayes baseline

nb_summary_path = (
    CLASSICAL_DIR / '01_naive_bayes'
    / 'results' / 'tables' / 'nb_run_summary.csv'
)

nb_summary = pd.read_csv(nb_summary_path).iloc[0]

comparison_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

nb_lr_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Naive Bayes': nb_summary[metric],
        'Logistic Regression': word_result['summary'][metric],
        'Difference': (
            word_result['summary'][metric]
            - nb_summary[metric]
        )
    }
    for metric in comparison_metrics
])

print("\nNaive Bayes vs Logistic Regression - Word TF-IDF")

display(nb_lr_comparison_df.round(6))

nb_lr_comparison_df.to_csv(
    TABLES_DIR / 'nb_vs_lr_word_comparison.csv',
    index=False
)

# Compare individual label performance

nb_per_label_df = pd.read_csv(
    CLASSICAL_DIR / '01_naive_bayes'
    / 'results' / 'tables'
    / 'nb_optimized_per_label_metrics.csv'
)

lr_per_label_df = word_result['optimized_per_label']

assert nb_per_label_df['Label'].tolist() == LABELS
assert lr_per_label_df['Label'].tolist() == LABELS

nb_lr_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'NB Average Precision':
        nb_per_label_df['Average Precision'],
    'LR Average Precision':
        lr_per_label_df['Average Precision'],
    'NB Optimized F1':
        nb_per_label_df['F1'],
    'LR Optimized F1':
        lr_per_label_df['F1']
})

nb_lr_per_label_df['AP Difference'] = (
    nb_lr_per_label_df['LR Average Precision']
    - nb_lr_per_label_df['NB Average Precision']
)

nb_lr_per_label_df['F1 Difference'] = (
    nb_lr_per_label_df['LR Optimized F1']
    - nb_lr_per_label_df['NB Optimized F1']
)

print("\nPer-label comparison")

display(nb_lr_per_label_df.round(6))

nb_lr_per_label_df.to_csv(
    TABLES_DIR / 'nb_vs_lr_word_per_label_comparison.csv',
    index=False
)

print("\nWord TF-IDF Logistic Regression summary")

print("Macro AP:",
      round(word_result['summary']['Macro AP'], 6))

print("Default Macro F1:",
      round(word_result['summary']['Default Macro F1'], 6))

print("Optimized Macro F1:",
      round(word_result['summary']['Optimized Macro F1'], 6))

print("Convergence warnings:",
      word_result['summary']['Convergence Warnings'])

print("Completed LR experiments:", len(lr_results))
print("Official test dataset: Not loaded")

In [ ]:
# Train and evaluate Logistic Regression using character TF-IDF

char_spec = experiment_specs[1]

assert char_spec['representation'] == 'character_tfidf'
assert char_spec['class_weight'] is None

char_result = run_lr_experiment(char_spec)

lr_results[char_spec['run_id']] = char_result

# Compare word and character representation performance

comparison_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

word_char_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Word TF-IDF': word_result['summary'][metric],
        'Character TF-IDF': char_result['summary'][metric],
        'Difference': (
            char_result['summary'][metric]
            - word_result['summary'][metric]
        )
    }
    for metric in comparison_metrics
])

print("\nWord vs Character TF-IDF performance")

display(word_char_comparison_df.round(6))

word_char_comparison_df.to_csv(
    TABLES_DIR / 'lr_word_vs_character_comparison.csv',
    index=False
)

# Compare ranking and optimized classification performance by label

word_metrics_df = word_result['optimized_per_label']
char_metrics_df = char_result['optimized_per_label']

assert word_metrics_df['Label'].tolist() == LABELS
assert char_metrics_df['Label'].tolist() == LABELS

word_char_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'Word AP': word_metrics_df['Average Precision'],
    'Character AP': char_metrics_df['Average Precision'],
    'Word F1': word_metrics_df['F1'],
    'Character F1': char_metrics_df['F1'],
    'Word Precision': word_metrics_df['Precision'],
    'Character Precision': char_metrics_df['Precision'],
    'Word Recall': word_metrics_df['Recall'],
    'Character Recall': char_metrics_df['Recall']
})

word_char_per_label_df['AP Difference'] = (
    word_char_per_label_df['Character AP']
    - word_char_per_label_df['Word AP']
)

word_char_per_label_df['F1 Difference'] = (
    word_char_per_label_df['Character F1']
    - word_char_per_label_df['Word F1']
)

print("\nPer-label representation comparison")

display(word_char_per_label_df.round(6))

word_char_per_label_df.to_csv(
    TABLES_DIR / 'lr_word_vs_character_per_label.csv',
    index=False
)

# Identify representation differences using validation metrics

ap_difference = (
    char_result['summary']['Macro AP']
    - word_result['summary']['Macro AP']
)

f1_difference = (
    char_result['summary']['Optimized Macro F1']
    - word_result['summary']['Optimized Macro F1']
)

print("\nCharacter TF-IDF experiment summary")

print("Macro AP:",
      round(char_result['summary']['Macro AP'], 6))

print("Macro AP difference vs word:",
      round(ap_difference, 6))

print("Default Macro F1:",
      round(char_result['summary']['Default Macro F1'], 6))

print("Optimized Macro F1:",
      round(char_result['summary']['Optimized Macro F1'], 6))

print("Optimized Macro F1 difference vs word:",
      round(f1_difference, 6))

print("Labels with improved AP:",
      int((word_char_per_label_df['AP Difference'] > 0).sum()),
      "/", len(LABELS))

print("Labels with improved F1:",
      int((word_char_per_label_df['F1 Difference'] > 0).sum()),
      "/", len(LABELS))

print("Convergence warnings:",
      char_result['summary']['Convergence Warnings'])

print("Completed LR experiments:", len(lr_results))
print("Official test dataset: Not loaded")

In [ ]:
# Train and evaluate Logistic Regression using combined TF-IDF

combined_spec = experiment_specs[2]

assert combined_spec['representation'] == 'combined_tfidf'
assert combined_spec['class_weight'] is None

combined_result = run_lr_experiment(combined_spec)

lr_results[combined_spec['run_id']] = combined_result

# Compare all three unweighted representations

representation_results = [
    word_result,
    char_result,
    combined_result
]

representation_comparison_df = pd.DataFrame([
    result['summary']
    for result in representation_results
])

assert representation_comparison_df['Training Samples'].nunique() == 1
assert representation_comparison_df['Validation Samples'].nunique() == 1
assert representation_comparison_df['Features'].nunique() == 1
assert representation_comparison_df['Class Weight'].eq('None').all()

comparison_columns = [
    'Feature Representation',
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

print("\nOverall representation comparison")

display(
    representation_comparison_df[
        comparison_columns
    ].round(6)
)

representation_comparison_df.to_csv(
    TABLES_DIR / 'lr_representation_comparison.csv',
    index=False
)

# Compare performance separately for each toxicity label

per_label_records = []

for result in representation_results:

    metrics_df = result['optimized_per_label'].copy()

    assert metrics_df['Label'].tolist() == LABELS

    metrics_df.insert(
        0,
        'Representation',
        result['spec']['representation']
    )

    per_label_records.append(metrics_df)

representation_per_label_df = pd.concat(
    per_label_records,
    ignore_index=True
)

representation_per_label_df.to_csv(
    TABLES_DIR / 'lr_representation_per_label_metrics.csv',
    index=False
)

# Create compact Average Precision and F1 comparison tables

ap_comparison_df = (
    representation_per_label_df.pivot(
        index='Label',
        columns='Representation',
        values='Average Precision'
    )
    .reindex(index=LABELS, columns=REPRESENTATIONS)
)

f1_comparison_df = (
    representation_per_label_df.pivot(
        index='Label',
        columns='Representation',
        values='F1'
    )
    .reindex(index=LABELS, columns=REPRESENTATIONS)
)

print("\nPer-label Average Precision comparison")
display(ap_comparison_df.round(6))

print("\nPer-label optimized F1 comparison")
display(f1_comparison_df.round(6))

ap_comparison_df.to_csv(
    TABLES_DIR / 'lr_representation_ap_comparison.csv'
)

f1_comparison_df.to_csv(
    TABLES_DIR / 'lr_representation_f1_comparison.csv'
)

# Measure combined feature performance against both alternatives

combined_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Combined AP': ap_comparison_df['combined_tfidf'].to_numpy(),
    'Combined AP vs Word': (
        ap_comparison_df['combined_tfidf']
        - ap_comparison_df['word_tfidf']
    ).to_numpy(),
    'Combined AP vs Character': (
        ap_comparison_df['combined_tfidf']
        - ap_comparison_df['character_tfidf']
    ).to_numpy(),
    'Combined F1': f1_comparison_df['combined_tfidf'].to_numpy(),
    'Combined F1 vs Word': (
        f1_comparison_df['combined_tfidf']
        - f1_comparison_df['word_tfidf']
    ).to_numpy(),
    'Combined F1 vs Character': (
        f1_comparison_df['combined_tfidf']
        - f1_comparison_df['character_tfidf']
    ).to_numpy()
})

print("\nCombined representation gains and losses")

display(combined_comparison_df.round(6))

combined_comparison_df.to_csv(
    TABLES_DIR / 'lr_combined_representation_effect.csv',
    index=False
)

# Rank the unweighted experiments using validation Macro AP

representation_ranking_df = (
    representation_comparison_df[
        [
            'Run ID',
            'Feature Representation',
            'Macro AP',
            'Optimized Macro F1',
            'Optimized Micro F1',
            'Training Time (s)'
        ]
    ]
    .sort_values(
        ['Macro AP', 'Optimized Macro F1'],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

representation_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(representation_ranking_df) + 1)
)

print("\nRepresentation ranking by validation Macro AP")

display(representation_ranking_df.round(6))

representation_ranking_df.to_csv(
    TABLES_DIR / 'lr_representation_ranking.csv',
    index=False
)

best_unweighted_run_id = representation_ranking_df.loc[
    0, 'Run ID'
]

best_unweighted_result = lr_results[
    best_unweighted_run_id
]

best_representation = best_unweighted_result[
    'spec'
]['representation']

print("\nRepresentation comparison summary")
print("Best unweighted run:", best_unweighted_run_id)
print("Selected representation:", best_representation)
print("Best validation Macro AP:",
      round(best_unweighted_result['summary']['Macro AP'], 6))
print("Best model optimized Macro F1:",
      round(best_unweighted_result['summary']['Optimized Macro F1'], 6))
print("Selection metric: Macro AP")
print("Completed LR experiments:", len(lr_results))
print("Official test dataset: Not loaded")

In [ ]:
# Visualize the controlled Logistic Regression representation experiments

from sklearn.metrics import precision_recall_curve

representation_names = {
    'word_tfidf': 'Word',
    'character_tfidf': 'Character',
    'combined_tfidf': 'Combined'
}

label_names = [
    label.replace('_', ' ').title()
    for label in LABELS
]

plot_colors = [
    REPRESENTATION_COLORS[representation]
    for representation in REPRESENTATIONS
]

plot_df = representation_comparison_df.set_index(
    'Feature Representation'
).loc[REPRESENTATIONS].copy()

plot_df.index = [
    representation_names[representation]
    for representation in plot_df.index
]

# Compare overall ranking and classification performance

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ranking_columns = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC'
]

classification_columns = [
    'Default Macro F1',
    'Optimized Macro F1',
    'Optimized Micro F1'
]

for ax, columns, title in [
    (axes[0], ranking_columns, 'Ranking Performance'),
    (axes[1], classification_columns, 'Classification Performance')
]:

    plot_df[columns].T.plot(
        kind='bar',
        color=plot_colors,
        width=0.75,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(title='Representation')

fig.suptitle(
    'Logistic Regression: TF-IDF Representation Comparison',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig(
    FIGURES_DIR / 'lr_representation_overall_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare per-label AP and optimized F1

fig, axes = plt.subplots(2, 1, figsize=(15, 11))

for ax, metrics_df, title in [
    (axes[0], ap_comparison_df, 'Per-Label Average Precision'),
    (axes[1], f1_comparison_df, 'Per-Label Optimized F1')
]:

    chart_df = metrics_df.copy()

    chart_df.index = label_names
    chart_df.columns = [
        representation_names[column]
        for column in chart_df.columns
    ]

    chart_df.plot(
        kind='bar',
        color=plot_colors,
        width=0.75,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.08)
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='x', visible=False)
    ax.legend(title='Representation')

fig.suptitle(
    'Logistic Regression: Per-Label Representation Performance',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'lr_representation_per_label_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Load saved validation scores without retraining models

representation_scores = {}

for result in representation_results:

    representation = result['spec']['representation']

    predictions_df = pd.read_parquet(
        result['predictions_path']
    )

    assert predictions_df['id'].tolist() == val_df['id'].tolist()

    saved_targets = predictions_df[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    assert np.array_equal(saved_targets, y_val)

    representation_scores[representation] = predictions_df[[
        f'{label}_score' for label in LABELS
    ]].to_numpy()

    assert np.isfinite(
        representation_scores[representation]
    ).all()


# Compare precision-recall curves for all representations

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for i, (ax, label) in enumerate(zip(axes.flat, LABELS)):

    for representation in REPRESENTATIONS:

        precision, recall, _ = precision_recall_curve(
            y_val[:, i],
            representation_scores[representation][:, i]
        )

        ap = ap_comparison_df.loc[label, representation]

        ax.step(
            recall,
            precision,
            where='post',
            linewidth=1.8,
            color=REPRESENTATION_COLORS[representation],
            label=(
                f'{representation_names[representation]} '
                f'(AP={ap:.4f})'
            )
        )

    ax.axhline(
        y_val[:, i].mean(),
        color='gray',
        linestyle='--',
        linewidth=1.2,
        label='Prevalence reference'
    )

    ax.set_title(label_names[i])
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

fig.suptitle(
    'Logistic Regression: Representation Precision-Recall Curves',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'lr_representation_precision_recall_curves.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare computational resource requirements

efficiency_records = []

for result in representation_results:

    representation = result['spec']['representation']

    feature_rows = feature_verification_df.loc[
        feature_verification_df['Representation']
        == representation
    ]

    efficiency_records.append({
        'Representation': representation,
        'Training Time (s)':
            result['summary']['Training Time (s)'],
        'Validation Prediction Time (s)':
            result['summary']['Validation Prediction Time (s)'],
        'Training Matrix Memory (MiB)': feature_rows.loc[
            feature_rows['Partition'] == 'train',
            'Sparse Memory (MiB)'
        ].iloc[0],
        'Model Size (MiB)':
            result['summary']['Model Size (MiB)']
    })

efficiency_df = pd.DataFrame(efficiency_records)

efficiency_df.to_csv(
    TABLES_DIR / 'lr_representation_efficiency.csv',
    index=False
)

display(efficiency_df.round(6))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, metric in zip(
    axes.flat,
    [
        'Training Time (s)',
        'Validation Prediction Time (s)',
        'Training Matrix Memory (MiB)',
        'Model Size (MiB)'
    ]
):

    bars = ax.bar(
        [
            representation_names[representation]
            for representation in efficiency_df['Representation']
        ],
        efficiency_df[metric],
        color=plot_colors,
        width=0.6
    )

    ax.bar_label(
        bars,
        fmt='%.3f',
        padding=4,
        fontsize=10
    )

    ax.set_title(metric)
    ax.set_ylim(0, efficiency_df[metric].max() * 1.2)
    ax.grid(axis='x', visible=False)

fig.suptitle(
    'Logistic Regression: Computational Efficiency',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'lr_representation_efficiency.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Document representation selection before class weighting

best_summary = best_unweighted_result['summary']

second_best = representation_ranking_df.iloc[1]

selection_config = {
    'selection_partition': 'validation',
    'selection_metric': 'macro_average_precision',
    'secondary_metric': 'optimized_macro_f1',
    'selected_run_id': best_unweighted_run_id,
    'selected_representation': best_representation,
    'selected_macro_ap': float(best_summary['Macro AP']),
    'selected_optimized_macro_f1':
        float(best_summary['Optimized Macro F1']),
    'macro_ap_gain_over_word': float(
        best_summary['Macro AP']
        - word_result['summary']['Macro AP']
    ),
    'macro_ap_gain_over_runner_up': float(
        best_summary['Macro AP']
        - second_best['Macro AP']
    ),
    'feature_budget': FEATURE_BUDGET,
    'statistical_significance_established': False,
    'next_experiment': 'balanced_class_weight',
    'official_test_used': False
}

with open(
    RUN_CONFIGS_DIR / 'lr_representation_selection.json',
    'w'
) as file:
    json.dump(selection_config, file, indent=4)

# Release temporary prediction arrays

del representation_scores
del predictions_df, saved_targets

gc.collect()

print("\nRepresentation selection summary")
print("Selected representation:", best_representation)
print("Selected run:", best_unweighted_run_id)
print("Validation Macro AP:",
      round(best_summary['Macro AP'], 6))
print("Optimized Macro F1:",
      round(best_summary['Optimized Macro F1'], 6))
print("Macro AP gain over word:",
      round(selection_config['macro_ap_gain_over_word'], 6))
print("Macro AP gain over runner-up:",
      round(selection_config['macro_ap_gain_over_runner_up'], 6))
print("Research figures generated: 4")
print("Next experiment: Balanced class weighting")
print("Official test dataset: Not used")

print("\nRepresentation comparison completed successfully.")

In [ ]:
# Train balanced Logistic Regression using the selected representation

weighted_spec = {
    'run_id': 'LR_COMBINED_BALANCED_S42',
    'representation': best_representation,
    'class_weight': 'balanced',
    'purpose': 'Controlled class imbalance handling experiment'
}

assert best_representation == 'combined_tfidf'
assert best_unweighted_result['spec']['class_weight'] is None

weighted_result = run_lr_experiment(weighted_spec)

lr_results[weighted_spec['run_id']] = weighted_result

# Compare weighted and unweighted overall performance

unweighted_summary = best_unweighted_result['summary']
weighted_summary = weighted_result['summary']

comparison_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

weighting_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Unweighted': unweighted_summary[metric],
        'Balanced': weighted_summary[metric],
        'Change': (
            weighted_summary[metric]
            - unweighted_summary[metric]
        )
    }
    for metric in comparison_metrics
])

print("\nUnweighted vs Balanced Logistic Regression")

display(weighting_comparison_df.round(6))

weighting_comparison_df.to_csv(
    TABLES_DIR / 'lr_class_weighting_comparison.csv',
    index=False
)

# Compare ranking and classification performance for each label

unweighted_per_label = (
    best_unweighted_result['optimized_per_label']
)

weighted_per_label = weighted_result['optimized_per_label']

assert unweighted_per_label['Label'].tolist() == LABELS
assert weighted_per_label['Label'].tolist() == LABELS

weighting_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'Actual Positives':
        unweighted_per_label['Positive Support'],
    'Unweighted AP':
        unweighted_per_label['Average Precision'],
    'Balanced AP':
        weighted_per_label['Average Precision'],
    'Unweighted Precision':
        unweighted_per_label['Precision'],
    'Balanced Precision':
        weighted_per_label['Precision'],
    'Unweighted Recall':
        unweighted_per_label['Recall'],
    'Balanced Recall':
        weighted_per_label['Recall'],
    'Unweighted F1':
        unweighted_per_label['F1'],
    'Balanced F1':
        weighted_per_label['F1']
})

for metric in ['AP', 'Precision', 'Recall', 'F1']:

    weighting_per_label_df[f'{metric} Change'] = (
        weighting_per_label_df[f'Balanced {metric}']
        - weighting_per_label_df[f'Unweighted {metric}']
    )

print("\nPer-label class weighting comparison")

display(weighting_per_label_df.round(6))

weighting_per_label_df.to_csv(
    TABLES_DIR / 'lr_class_weighting_per_label.csv',
    index=False
)

# Compare classification error counts at optimized thresholds

weighting_errors_df = pd.DataFrame({
    'Label': LABELS,
    'Unweighted TP': unweighted_per_label['TP'],
    'Balanced TP': weighted_per_label['TP'],
    'Unweighted FP': unweighted_per_label['FP'],
    'Balanced FP': weighted_per_label['FP'],
    'Unweighted FN': unweighted_per_label['FN'],
    'Balanced FN': weighted_per_label['FN'],
    'Unweighted TN': unweighted_per_label['TN'],
    'Balanced TN': weighted_per_label['TN']
})

for error_type in ['TP', 'FP', 'FN', 'TN']:

    weighting_errors_df[f'{error_type} Change'] = (
        weighting_errors_df[f'Balanced {error_type}']
        - weighting_errors_df[f'Unweighted {error_type}']
    )

assert np.array_equal(
    unweighted_per_label['Positive Support'],
    weighted_per_label['Positive Support']
)

assert np.array_equal(
    unweighted_per_label['Negative Support'],
    weighted_per_label['Negative Support']
)

print("\nClassification error comparison")

display(weighting_errors_df)

weighting_errors_df.to_csv(
    TABLES_DIR / 'lr_class_weighting_error_comparison.csv',
    index=False
)

# Compare the selected decision thresholds

weighting_threshold_df = pd.DataFrame({
    'Label': LABELS,
    'Unweighted Threshold':
        unweighted_per_label['Threshold'],
    'Balanced Threshold':
        weighted_per_label['Threshold']
})

weighting_threshold_df['Threshold Change'] = (
    weighting_threshold_df['Balanced Threshold']
    - weighting_threshold_df['Unweighted Threshold']
)

print("\nOptimized threshold comparison")

display(weighting_threshold_df.round(8))

weighting_threshold_df.to_csv(
    TABLES_DIR / 'lr_class_weighting_threshold_comparison.csv',
    index=False
)

# Examine rare-label performance separately

RARE_LABELS = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

rare_label_comparison_df = weighting_per_label_df.loc[
    weighting_per_label_df['Label'].isin(RARE_LABELS)
].copy()

print("\nRare-label performance comparison")

display(rare_label_comparison_df.round(6))

rare_label_comparison_df.to_csv(
    TABLES_DIR / 'lr_class_weighting_rare_labels.csv',
    index=False
)

# Summarize the weighting experiment

print("\nClass weighting experiment summary")

print("Unweighted Macro AP:",
      round(unweighted_summary['Macro AP'], 6))

print("Balanced Macro AP:",
      round(weighted_summary['Macro AP'], 6))

print("Macro AP change:",
      round(weighted_summary['Macro AP']
            - unweighted_summary['Macro AP'], 6))

print("Unweighted optimized Macro F1:",
      round(unweighted_summary['Optimized Macro F1'], 6))

print("Balanced optimized Macro F1:",
      round(weighted_summary['Optimized Macro F1'], 6))

print("Labels with improved AP:",
      int((weighting_per_label_df['AP Change'] > 0).sum()),
      "/", len(LABELS))

print("Labels with improved F1:",
      int((weighting_per_label_df['F1 Change'] > 0).sum()),
      "/", len(LABELS))

print("Unweighted rare-label mean AP:",
      round(rare_label_comparison_df['Unweighted AP'].mean(), 6))

print("Balanced rare-label mean AP:",
      round(rare_label_comparison_df['Balanced AP'].mean(), 6))

print("Unweighted rare-label mean F1:",
      round(rare_label_comparison_df['Unweighted F1'].mean(), 6))

print("Balanced rare-label mean F1:",
      round(rare_label_comparison_df['Balanced F1'].mean(), 6))

print("Convergence warnings:",
      weighted_summary['Convergence Warnings'])

print("Completed LR experiments:", len(lr_results))
print("Official test dataset: Not loaded")

print("\nClass weighting experiment completed successfully.")

In [ ]:
# Visualize the effect of class weighting

weighting_colors = ['#4C78A8', '#54A24B']

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for ax, metric in zip(
    axes.flat,
    ['AP', 'Precision', 'Recall', 'F1']
):

    columns = [
        f'Unweighted {metric}',
        f'Balanced {metric}'
    ]

    chart_df = weighting_per_label_df.set_index(
        'Label'
    )[columns].copy()

    chart_df.index = label_names
    chart_df.columns = ['Unweighted', 'Balanced']

    chart_df.plot(
        kind='bar',
        ax=ax,
        color=weighting_colors,
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    title = (
        'Average Precision'
        if metric == 'AP'
        else f'Optimized {metric}'
    )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend()

fig.suptitle(
    'Logistic Regression: Effect of Balanced Class Weighting',
    fontsize=17
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'lr_class_weighting_per_label_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare rare-label classification error counts

rare_errors_df = weighting_errors_df.loc[
    weighting_errors_df['Label'].isin(RARE_LABELS)
].set_index('Label')

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

for ax, error_type, title in zip(
    axes,
    ['TP', 'FP', 'FN'],
    ['True Positives', 'False Positives', 'False Negatives']
):

    chart_df = rare_errors_df[[
        f'Unweighted {error_type}',
        f'Balanced {error_type}'
    ]].copy()

    chart_df.index = [
        label.replace('_', ' ').title()
        for label in chart_df.index
    ]

    chart_df.columns = ['Unweighted', 'Balanced']

    chart_df.plot(
        kind='bar',
        ax=ax,
        color=weighting_colors,
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%d',
            padding=3,
            fontsize=10
        )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.set_ylim(0, chart_df.to_numpy().max() * 1.2)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend()

fig.suptitle(
    'Logistic Regression: Rare-Label Errors at Optimized Thresholds',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.92])

fig.savefig(
    FIGURES_DIR / 'lr_class_weighting_rare_label_errors.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Consolidate all completed Logistic Regression experiments

lr_comparison_df = pd.DataFrame([
    result['summary']
    for result in lr_results.values()
])

assert len(lr_comparison_df) == 4
assert lr_comparison_df['Run ID'].is_unique
assert lr_comparison_df['Training Samples'].nunique() == 1
assert lr_comparison_df['Validation Samples'].nunique() == 1
assert lr_comparison_df['Features'].nunique() == 1

assert lr_comparison_df['Convergence Warnings'].eq(0).all()
assert lr_comparison_df['Status'].eq('Completed').all()
assert not lr_comparison_df['Official Test Used'].any()

lr_comparison_df.to_csv(
    TABLES_DIR / 'lr_all_experiments_comparison.csv',
    index=False
)

run_display_names = {
    'LR_WORD_BASELINE_S42': 'Word',
    'LR_CHAR_BASELINE_S42': 'Character',
    'LR_COMBINED_BASELINE_S42': 'Combined',
    'LR_COMBINED_BALANCED_S42': 'Combined Balanced'
}

lr_plot_df = lr_comparison_df.copy()

lr_plot_df['Configuration'] = lr_plot_df[
    'Run ID'
].map(run_display_names)

lr_plot_df = lr_plot_df.set_index('Configuration')

# Compare all four configurations

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

comparison_colors = [
    '#4C78A8',
    '#F58518',
    '#54A24B',
    '#B279A2'
]

for ax, metrics, title in [
    (
        axes[0],
        ['Macro AP', 'Micro AP', 'Macro ROC-AUC'],
        'Ranking Performance'
    ),
    (
        axes[1],
        [
            'Default Macro F1',
            'Optimized Macro F1',
            'Optimized Micro F1'
        ],
        'Classification Performance'
    )
]:

    lr_plot_df[metrics].T.plot(
        kind='bar',
        ax=ax,
        color=comparison_colors,
        width=0.8
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.13)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(fontsize=9)

fig.suptitle(
    'Logistic Regression: Final Configuration Comparison',
    fontsize=17
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig(
    FIGURES_DIR / 'lr_final_configuration_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Rank all configurations using validation Macro AP

lr_final_ranking_df = (
    lr_comparison_df[[
        'Run ID',
        'Feature Representation',
        'Class Weight',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)'
    ]]
    .sort_values(
        ['Macro AP', 'Optimized Macro F1'],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

lr_final_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(lr_final_ranking_df) + 1)
)

print("Final Logistic Regression experiment ranking")

display(lr_final_ranking_df.round(6))

lr_final_ranking_df.to_csv(
    TABLES_DIR / 'lr_final_experiment_ranking.csv',
    index=False
)

best_lr_run_id = lr_final_ranking_df.loc[0, 'Run ID']

best_lr_result = lr_results[best_lr_run_id]
best_lr_summary = best_lr_result['summary']

# Record the selected Logistic Regression candidate

lr_final_selection = {
    'model': MODEL_NAME,
    'selection_partition': 'validation',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'selected_run_id': best_lr_run_id,
    'selected_representation':
        best_lr_result['spec']['representation'],
    'selected_class_weight':
        best_lr_result['spec']['class_weight'],
    'validation_macro_ap':
        float(best_lr_summary['Macro AP']),
    'validation_macro_roc_auc':
        float(best_lr_summary['Macro ROC-AUC']),
    'validation_default_macro_f1':
        float(best_lr_summary['Default Macro F1']),
    'validation_optimized_macro_f1':
        float(best_lr_summary['Optimized Macro F1']),
    'validation_optimized_micro_f1':
        float(best_lr_summary['Optimized Micro F1']),
    'model_path': str(
        best_lr_result['model_path'].relative_to(PROJECT_ROOT)
    ),
    'threshold_path': str(
        (
            RUN_CONFIGS_DIR
            / f'{best_lr_run_id}_thresholds.json'
        ).relative_to(PROJECT_ROOT)
    ),
    'predictions_path': str(
        best_lr_result['predictions_path'].relative_to(PROJECT_ROOT)
    ),
    'selection_scope': 'Logistic Regression configurations only',
    'statistical_significance_established': False,
    'final_project_model_frozen': False,
    'official_test_used': False
}

LR_FINAL_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'lr_final_selection.json'
)

with open(LR_FINAL_SELECTION_PATH, 'w') as file:
    json.dump(lr_final_selection, file, indent=4)

print("\nLogistic Regression selection summary")

print("Selected run:", best_lr_run_id)

print("Selected representation:",
      lr_final_selection['selected_representation'])

print("Selected class weight:",
      lr_final_selection['selected_class_weight'])

print("Validation Macro AP:",
      round(best_lr_summary['Macro AP'], 6))

print("Optimized Macro F1:",
      round(best_lr_summary['Optimized Macro F1'], 6))

print("Optimized Micro F1:",
      round(best_lr_summary['Optimized Micro F1'], 6))

print("Completed LR experiments:", len(lr_results))
print("Additional research figures generated: 3")
print("Final project model frozen: No")
print("Official test dataset: Not used")

print("\nLogistic Regression comparison completed successfully.")

In [ ]:
# Verify completed Logistic Regression runs before registration

assert len(lr_results) == 4

registry_records = []
registration_checks = []

for run_id, result in lr_results.items():

    assert result['summary']['Status'] == 'Completed'
    assert result['summary']['Convergence Warnings'] == 0
    assert not result['summary']['Official Test Used']

    run_dir = result['results_path']

    manifest_df = pd.read_csv(
        run_dir / 'tables' / 'artifact_manifest.csv'
    )

    assert manifest_df['Run ID'].eq(run_id).all()

    # Recheck fingerprints of saved experiment artifacts

    for _, artifact in manifest_df.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file()
        assert path.stat().st_size == artifact['Size Bytes']

        assert (
            calculate_sha256(path)
            == artifact['SHA256']
        ), f"Artifact fingerprint mismatch: {path}"

    record = result['registry_record'].copy()

    assert record['run_id'] == run_id

    assert (PROJECT_ROOT / record['model_path']).is_file()
    assert (PROJECT_ROOT / record['config_path']).is_file()
    assert (PROJECT_ROOT / record['results_path']).is_dir()

    # Identify the selected Logistic Regression configuration

    if run_id == best_lr_run_id:

        record['notes'] += (
            ' Selected Logistic Regression configuration '
            'based on validation Macro AP.'
        )

    registry_records.append(record)

    registration_checks.append({
        'Run ID': run_id,
        'Fingerprinted Artifacts Verified': len(manifest_df),
        'Convergence Verified': True,
        'Model Reconstruction Verified': True,
        'Official Test Used': False,
        'Selected LR Configuration': run_id == best_lr_run_id,
        'Status': 'Verified'
    })

registration_verification_df = pd.DataFrame(
    registration_checks
)

display(registration_verification_df)

registration_verification_df.to_csv(
    TABLES_DIR / 'lr_registration_verification.csv',
    index=False
)

# Load the shared experiment registry

registry_df = pd.read_csv(
    REGISTRY_PATH,
    keep_default_na=False
)

assert registry_df['run_id'].is_unique

for record in registry_records:

    assert set(record.keys()) == set(registry_df.columns)

# Preserve previous experiments and prevent duplicate run IDs

lr_run_ids = [
    record['run_id']
    for record in registry_records
]

previous_registry_df = registry_df.loc[
    ~registry_df['run_id'].isin(lr_run_ids)
].copy()

new_registry_df = pd.DataFrame(
    registry_records,
    columns=registry_df.columns
)

updated_registry_df = pd.concat(
    [previous_registry_df, new_registry_df],
    ignore_index=True
)

assert updated_registry_df['run_id'].is_unique

assert 'NB_WORD_BASELINE_S42' in (
    updated_registry_df['run_id'].tolist()
)

assert len(updated_registry_df) == (
    len(previous_registry_df) + len(registry_records)
)

# Save the updated registry

temporary_registry_path = REGISTRY_PATH.with_suffix('.tmp')

updated_registry_df.to_csv(
    temporary_registry_path,
    index=False
)

temporary_registry_path.replace(REGISTRY_PATH)

# Reload and verify the registered experiments

saved_registry_df = pd.read_csv(
    REGISTRY_PATH,
    keep_default_na=False,
    float_precision='round_trip'
)

assert saved_registry_df['run_id'].is_unique

for run_id, result in lr_results.items():

    saved_record = saved_registry_df.loc[
        saved_registry_df['run_id'] == run_id
    ].iloc[0]

    assert saved_record['status'] == 'Completed'

    assert np.isclose(
        float(saved_record['validation_macro_ap']),
        result['summary']['Macro AP'],
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        float(saved_record['validation_macro_f1']),
        result['summary']['Optimized Macro F1'],
        rtol=0,
        atol=1e-12
    )

    assert int(saved_record['training_samples']) == len(y_train)

    assert (PROJECT_ROOT / saved_record['model_path']).is_file()
    assert (PROJECT_ROOT / saved_record['config_path']).is_file()

# Verify that previous experiment records remain unchanged

preserved_registry_df = saved_registry_df.loc[
    ~saved_registry_df['run_id'].isin(lr_run_ids)
]

pd.testing.assert_frame_equal(
    previous_registry_df.reset_index(drop=True).astype(str),
    preserved_registry_df.reset_index(drop=True).astype(str)
)

# Display all registered experiments

registry_display_columns = [
    'run_id',
    'model',
    'feature_representation',
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds',
    'status'
]

print("\nUpdated shared experiment registry")

display(
    saved_registry_df[
        registry_display_columns
    ].round(6)
)

# Create the current validation leaderboard

leaderboard_df = saved_registry_df[
    registry_display_columns
].copy()

leaderboard_df['validation_macro_ap'] = pd.to_numeric(
    leaderboard_df['validation_macro_ap']
)

leaderboard_df['validation_macro_f1'] = pd.to_numeric(
    leaderboard_df['validation_macro_f1']
)

leaderboard_df = leaderboard_df.sort_values(
    ['validation_macro_ap', 'validation_macro_f1'],
    ascending=[False, False]
).reset_index(drop=True)

leaderboard_df.insert(
    0,
    'Rank',
    np.arange(1, len(leaderboard_df) + 1)
)

print("\nCurrent validation leaderboard")

display(leaderboard_df.round(6))

leaderboard_df.to_csv(
    TABLES_DIR / 'lr_current_validation_leaderboard.csv',
    index=False
)

print("\nExperiment registration summary")

print("Previously registered experiments:",
      len(previous_registry_df))

print("Logistic Regression experiments registered:",
      len(registry_records))

print("Total registered experiments:",
      len(saved_registry_df))

print("Duplicate run IDs:",
      saved_registry_df['run_id'].duplicated().sum())

print("Critical LR artifact fingerprints verified:",
      registration_verification_df[
          'Fingerprinted Artifacts Verified'
      ].sum())

print("Previous experiment records: Preserved")
print("Selected Logistic Regression run:", best_lr_run_id)
print("Registry verification: Passed")
print("Official test dataset: Not used")

print("\nLogistic Regression experiments registered successfully.")

In [ ]:
# Verify notebook-level analysis artifacts

required_tables = [
    'environment_versions.csv',
    'environment_compatibility.csv',
    'dataset_verification.csv',
    'feature_representation_verification.csv',
    'label_support_and_imbalance.csv',
    'lr_experiment_plan.csv',
    'lr_hyperparameter_rationale.csv',
    'lr_expected_balanced_class_weights.csv',
    'nb_vs_lr_word_comparison.csv',
    'nb_vs_lr_word_per_label_comparison.csv',
    'lr_word_vs_character_comparison.csv',
    'lr_word_vs_character_per_label.csv',
    'lr_representation_comparison.csv',
    'lr_representation_per_label_metrics.csv',
    'lr_representation_ap_comparison.csv',
    'lr_representation_f1_comparison.csv',
    'lr_combined_representation_effect.csv',
    'lr_representation_ranking.csv',
    'lr_representation_efficiency.csv',
    'lr_class_weighting_comparison.csv',
    'lr_class_weighting_per_label.csv',
    'lr_class_weighting_error_comparison.csv',
    'lr_class_weighting_threshold_comparison.csv',
    'lr_class_weighting_rare_labels.csv',
    'lr_all_experiments_comparison.csv',
    'lr_final_experiment_ranking.csv',
    'lr_registration_verification.csv',
    'lr_current_validation_leaderboard.csv'
]

required_figures = [
    'lr_representation_overall_comparison.png',
    'lr_representation_per_label_comparison.png',
    'lr_representation_precision_recall_curves.png',
    'lr_representation_efficiency.png',
    'lr_class_weighting_per_label_comparison.png',
    'lr_class_weighting_rare_label_errors.png',
    'lr_final_configuration_comparison.png'
]

required_configs = [
    'lr_experiment_config.json',
    'lr_representation_selection.json',
    'lr_final_selection.json'
]

verification_records = []

def verify_artifact(path, category, run_id='Shared'):

    available = path.is_file() and path.stat().st_size > 0

    verification_records.append({
        'Run ID': run_id,
        'Type': category,
        'Filename': path.name,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size if available else 0,
        'Status': 'Available' if available else 'Missing'
    })


for filename in required_tables:
    verify_artifact(TABLES_DIR / filename, 'Analysis Table')

for filename in required_figures:
    verify_artifact(FIGURES_DIR / filename, 'Figure')

for filename in required_configs:
    verify_artifact(RUN_CONFIGS_DIR / filename, 'Configuration')


# Verify individual experiment outputs

run_required_tables = [
    'feature_verification.csv',
    'classifier_training_summary.csv',
    'default_overall_metrics.csv',
    'optimized_overall_metrics.csv',
    'default_per_label_metrics.csv',
    'optimized_per_label_metrics.csv',
    'optimized_thresholds.csv',
    'overall_threshold_comparison.csv',
    'per_label_threshold_comparison.csv',
    'run_summary.csv',
    'artifact_manifest.csv'
]

for run_id, result in lr_results.items():

    run_dir = result['results_path']

    for filename in run_required_tables:

        verify_artifact(
            run_dir / 'tables' / filename,
            'Run Table',
            run_id
        )

    verify_artifact(
        result['model_path'],
        'Model',
        run_id
    )

    verify_artifact(
        result['predictions_path'],
        'Prediction',
        run_id
    )

    verify_artifact(
        RUN_CONFIGS_DIR / f'{run_id}.json',
        'Configuration',
        run_id
    )

    verify_artifact(
        RUN_CONFIGS_DIR / f'{run_id}_thresholds.json',
        'Configuration',
        run_id
    )

    for filename in [
        'training_log.json',
        'training_warnings.json'
    ]:

        verify_artifact(
            run_dir / 'logs' / filename,
            'Training Log',
            run_id
        )

    # Recheck critical artifact fingerprints

    manifest_df = pd.read_csv(
        run_dir / 'tables' / 'artifact_manifest.csv'
    )

    for _, artifact in manifest_df.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file()
        assert calculate_sha256(path) == artifact['SHA256']


# Verify fitted vectorizer dependencies

vectorizer_records = feature_manifest.loc[
    feature_manifest['Artifact Type'] == 'Vectorizer'
]

assert len(vectorizer_records) == 4

for _, record in vectorizer_records.iterrows():

    path = PROJECT_ROOT / record['Path']

    assert calculate_sha256(path) == record['SHA256']

    verify_artifact(path, 'Shared Dependency')

verify_artifact(
    SRC_DIR / 'evaluation_utils.py',
    'Shared Dependency'
)

verify_artifact(
    FEATURES_DIR / 'feature_artifact_manifest.csv',
    'Shared Dependency'
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

final_verification_df = pd.DataFrame(
    verification_records
)

final_verification_df.to_csv(
    TABLES_DIR / 'lr_final_artifact_verification.csv',
    index=False
)

assert final_verification_df['Status'].eq('Available').all()

print("Final artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)


# Verify the final selection and experiment registration

with open(LR_FINAL_SELECTION_PATH, 'r') as file:
    saved_selection = json.load(file)

assert saved_selection['selected_run_id'] == best_lr_run_id
assert not saved_selection['final_project_model_frozen']
assert not saved_selection['official_test_used']

assert np.isclose(
    saved_selection['validation_macro_ap'],
    best_lr_summary['Macro AP'],
    rtol=0,
    atol=1e-12
)

saved_registry_df = pd.read_csv(REGISTRY_PATH)

assert saved_registry_df['run_id'].is_unique
assert set(lr_results).issubset(set(saved_registry_df['run_id']))


# Record evidence-based methodological decisions

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(DECISION_LOG_PATH)

lr_decisions = [
    {
        'decision_id': 'D027',
        'component': 'Logistic Regression configuration',
        'decision': (
            'L2-regularized Logistic Regression with '
            'liblinear solver and C=1.0'
        ),
        'justification': (
            'Provides a consistent sparse binary classification '
            'baseline. All four experiments converged without '
            'warnings. Hyperparameters were held constant during '
            'controlled comparisons.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D028',
        'component': 'Representation comparison',
        'decision': (
            'Compare word, character and combined TF-IDF '
            'using a fixed 50000-feature budget'
        ),
        'justification': (
            'Controls total feature dimensionality, training '
            'samples and classifier configuration while '
            'investigating representation effectiveness.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D029',
        'component': 'Logistic Regression representation selection',
        'decision': (
            f'Select {best_representation} for the '
            'controlled weighting experiment'
        ),
        'justification': (
            f"Combined TF-IDF achieves validation Macro AP "
            f"of {combined_result['summary']['Macro AP']:.6f}, "
            f"compared with "
            f"{char_result['summary']['Macro AP']:.6f} for "
            f"character and "
            f"{word_result['summary']['Macro AP']:.6f} for word. "
            'The improvement is descriptive and statistical '
            'significance has not been established.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D030',
        'component': 'Class imbalance handling',
        'decision': (
            'Compare balanced class weighting against '
            'unweighted Logistic Regression'
        ),
        'justification': (
            f"Weighting improves Macro AP from "
            f"{unweighted_summary['Macro AP']:.6f} to "
            f"{weighted_summary['Macro AP']:.6f}. "
            'Rare-label detection improves overall without '
            'changing training sample composition. '
            'Micro AP and severe toxicity performance decline, '
            'demonstrating label-specific trade-offs.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D031',
        'component': 'Logistic Regression model selection',
        'decision': f'Select {best_lr_run_id}',
        'justification': (
            f"Highest validation Macro AP of "
            f"{best_lr_summary['Macro AP']:.6f} and "
            f"optimized Macro F1 of "
            f"{best_lr_summary['Optimized Macro F1']:.6f} "
            'among the four evaluated LR configurations. '
            'Selection is restricted to Logistic Regression '
            'and does not freeze the final project model.'
        ),
        'status': 'Selected'
    },
    {
        'decision_id': 'D032',
        'component': 'Logistic Regression experiment scope',
        'decision': (
            'Retain four controlled Logistic Regression runs '
            'without additional regularization tuning'
        ),
        'justification': (
            'Representation, weighting and threshold effects '
            'have been investigated using a fixed configuration. '
            'Additional hyperparameter searches are deprioritized '
            'to complete the remaining classical models and '
            'required transformer experiments.'
        ),
        'status': 'Implemented'
    }
]

for record in lr_decisions:

    mask = decisions_df['decision_id'].eq(
        record['decision_id']
    )

    if mask.any():

        assert decisions_df.loc[
            mask, 'component'
        ].eq(record['component']).all()

        for column, value in record.items():
            decisions_df.loc[mask, column] = value

    else:

        decisions_df = pd.concat(
            [decisions_df, pd.DataFrame([record])],
            ignore_index=True
        )

assert decisions_df['decision_id'].is_unique

decisions_df.to_csv(
    DECISION_LOG_PATH,
    index=False
)

print("\nNew methodological decisions")

display(
    decisions_df.loc[
        decisions_df['decision_id'].isin([
            record['decision_id']
            for record in lr_decisions
        ])
    ][['decision_id', 'component', 'decision', 'status']]
)


# Construct the final experiment results table

summary_table = [
    '| Configuration | Macro AP | Macro ROC-AUC | Default Macro F1 | Optimized Macro F1 | Optimized Micro F1 |',
    '|---|---:|---:|---:|---:|---:|'
]

for _, row in lr_final_ranking_df.iterrows():

    summary_table.append(
        f"| {run_display_names[row['Run ID']]} "
        f"| {row['Macro AP']:.6f} "
        f"| {row['Macro ROC-AUC']:.6f} "
        f"| {row['Default Macro F1']:.6f} "
        f"| {row['Optimized Macro F1']:.6f} "
        f"| {row['Optimized Micro F1']:.6f} |"
    )

summary_table_text = '\n'.join(summary_table)


# Save the final notebook summary

summary = f"""# Notebook 03.02 - Logistic Regression

## Objective

Compare Logistic Regression with the Naive Bayes baseline,
investigate word and character feature representations,
evaluate balanced class weighting and optimize individual
classification thresholds.

## Experimental Configuration

- Training samples: {len(y_train):,}
- Validation samples: {len(y_val):,}
- Features per representation: {FEATURE_BUDGET:,}
- Classification strategy: One-vs-Rest
- Solver: liblinear
- Regularization: L2
- C: 1.0
- Random seed: {SEED}
- Completed experiments: {len(lr_results)}

## Validation Results

{summary_table_text}

## Research Findings

### Learning algorithm comparison

Unweighted word-based Logistic Regression outperforms
Multinomial Naive Bayes using identical datasets and
word TF-IDF features.

Macro AP improves from {nb_summary['Macro AP']:.6f}
to {word_result['summary']['Macro AP']:.6f}.

### Feature representation

Character features improve Macro AP over word features.
Combined TF-IDF provides the highest unweighted Macro AP
under the fixed total feature budget.

Character-only features remain stronger for identity hate
than the combined unweighted representation.

### Class weighting

Balanced class weighting improves Macro AP from
{unweighted_summary['Macro AP']:.6f}
to {weighted_summary['Macro AP']:.6f}.

Optimized Macro F1 improves from
{unweighted_summary['Optimized Macro F1']:.6f}
to {weighted_summary['Optimized Macro F1']:.6f}.

Rare-label mean AP improves from
{rare_label_comparison_df['Unweighted AP'].mean():.6f}
to {rare_label_comparison_df['Balanced AP'].mean():.6f}.

Rare-label mean F1 improves from
{rare_label_comparison_df['Unweighted F1'].mean():.6f}
to {rare_label_comparison_df['Balanced F1'].mean():.6f}.

Threat detection improves from 50 to 62 true positives,
while false positives decrease from 48 to 42.

Micro AP decreases after weighting, and severe toxicity
shows a small reduction in AP and optimized F1.

### Threshold optimization

Per-label validation threshold optimization improves
Macro F1 for all four Logistic Regression configurations.

Threshold optimization does not change ranking metrics.

## Selected Logistic Regression Configuration

- Run ID: {best_lr_run_id}
- Representation: {best_lr_result['spec']['representation']}
- Class weighting: {best_lr_result['spec']['class_weight']}
- Validation Macro AP: {best_lr_summary['Macro AP']:.6f}
- Validation Macro ROC-AUC: {best_lr_summary['Macro ROC-AUC']:.6f}
- Optimized Macro F1: {best_lr_summary['Optimized Macro F1']:.6f}
- Optimized Micro F1: {best_lr_summary['Optimized Micro F1']:.6f}
- Training time: {best_lr_summary['Training Time (s)']:.6f} seconds

## Reproducibility

- All four experiments completed.
- All 24 binary classifiers passed convergence checks.
- All saved models reproduced their validation scores.
- Prediction files and threshold configurations verified.
- Model and dependency fingerprints verified.
- All four experiments registered in the shared registry.

## Limitations

- Results are based on validation data.
- Optimized validation F1 may be optimistic.
- Statistical significance has not been established.
- Class-weighted scores are not necessarily calibrated
  probabilities under natural dataset prevalence.
- Fixed feature budgets do not establish the optimal
  vocabulary size or optimal feature allocation.
- Regularization strength has not been individually tuned.
- Runtime measurements are single-run observations.
- The comparison does not establish independent test performance.

## Evaluation Protocol

Macro Average Precision remains the primary model selection
criterion. Official test data has not been used for fitting,
threshold optimization or model selection.

The selected configuration is the leading Logistic Regression
candidate only. The final project model is not frozen.

## Next Stage

Notebook 03.03 - Linear SVM.
"""

summary_path = RESULTS_DIR / 'summary.md'

summary_path.write_text(
    summary,
    encoding='utf-8'
)

assert summary_path.is_file()
assert summary_path.stat().st_size > 0

print("\nLogistic Regression completion summary")

print("Notebook-level analysis tables:", len(required_tables))
print("Research figures:", len(required_figures))
print("Completed LR experiments:", len(lr_results))
print("Total required artifact entries:", len(final_verification_df))
print("Missing artifacts:",
      final_verification_df['Status'].eq('Missing').sum())
print("Total methodological decisions:", len(decisions_df))
print("Total registered experiments:", len(saved_registry_df))
print("Selected LR configuration:", best_lr_run_id)
print("Notebook summary saved.")
print("Official test dataset: Not used")
print("Final project model frozen: No")

print("\nLogistic Regression notebook completed successfully.")